In [ ]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [ ]:
%run ./00_setup_parameters 

In [ ]:
# Helper to execute multi-statement SQL files dynamically
def run_sql_file(file_path):
    print(f"Executing SQL file: {file_path}")
    with open(file_path, "r") as f:
        sql_text = f.read()
    for stmt in sql_text.split(";"):
        if stmt.strip():
            formatted = (
                stmt.replace("${catalog}", catalog)
                .replace("${schema}", schema)
            )
            spark.sql(formatted)

# COMMAND ----------
# Step 1: Initialize Catalog and Schemas
run_sql_file("./00_init_catalog_schema.sql")

# COMMAND ----------
# Step 2: Execute Layer DDLs
run_sql_file("./01_bronze_ddl.sql")
run_sql_file("./02_silver_ddl.sql")
run_sql_file("./03_gold_ddl.sql")

# COMMAND ----------
# Step 3: Run Python Dimension Populators

print(f"catalog: {catalog}")
print(f"schema: {schema}")
import build_date_dimension
import build_time_dimension

print("Populating Date Dimension...")
build_date_dimension.generate_date_dim(spark=spark, catalog=f"{catalog}", schema=f"{schema}")
print("Populating Time Dimension...")
build_time_dimension.generate_time_dim(spark=spark,catalog=f"{catalog}", schema=f"{schema}")


print("Environment setup and dimensions successfully completed!")